# Synthetic integrated-discrepancy calibration demo

This notebook demonstrates the packaged `integrated_delta_calibration` API using the checked-in `syntheticProb` dataset. It:

1. loads simulator and observation data,
2. fits a Gaussian-process emulator to simulator outputs,
3. runs the integrated-discrepancy sampler, and
4. compares the emulator prediction with the calibrated prediction.

The sampler settings below are intentionally small so the notebook is suitable for a quick demonstration. Increase `N_MCMC` and `BURN_IN` for scientific use.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF

from integrated_delta_calibration import eta_predict, run_delta_sampler

plt.style.use("seaborn-v0_8-whitegrid")
DATA_ROOT = Path("syntheticProb")
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        "Run this notebook from the repository root so syntheticProb/ is available."
    )

MODEL_ROOT = DATA_ROOT / "modelData"
OBS_ROOT = DATA_ROOT / "observationData"


## Load the synthetic data

The simulator files contain application-domain values, two calibration parameters, and simulator responses. The observation files contain application-domain values and measured responses. The files are whitespace-delimited and include one header row.

In [ ]:
x_model = np.loadtxt(MODEL_ROOT / "appDomain.txt", skiprows=1)
theta_model = np.loadtxt(MODEL_ROOT / "thetaVals.txt", skiprows=1)
y_model = np.loadtxt(MODEL_ROOT / "modelPredictions.txt", skiprows=1)

x_obs = np.loadtxt(OBS_ROOT / "appDomain.txt", skiprows=1)
y_obs = np.loadtxt(OBS_ROOT / "observationData.txt", skiprows=1)

print(f"Simulator inputs: {theta_model.shape[0]} runs")
print(f"Simulator parameter dimension: {theta_model.shape[1]}")
print(f"Observations: {y_obs.size} points")
print(f"Parameter ranges:\n{np.column_stack((theta_model.min(axis=0), theta_model.max(axis=0)))}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_model, y_model, ".", ms=2, alpha=0.25, label="Simulator runs")
ax.plot(x_obs, y_obs, "o", ms=4, label="Synthetic observations")
ax.set(xlabel="Application domain", ylabel="Response", title="Synthetic calibration data")
ax.legend()
plt.show()


## Fit the Gaussian-process emulator

The emulator input is the joint vector `(x, theta_1, theta_2)`. The package sampler expects an object with the scikit-learn `GaussianProcessRegressor.predict(..., return_std=True)` interface, so a fitted scikit-learn GP can be passed directly.

In [ ]:
emulator_inputs = np.column_stack((x_model, theta_model))
kernel = ConstantKernel(1.0, (1e-2, 1e2)) * RBF(
    length_scale=np.ones(emulator_inputs.shape[1]),
    length_scale_bounds=(1e-2, 1e2),
)
emulator = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    normalize_y=True,
    n_restarts_optimizer=1,
    random_state=7,
)
emulator.fit(emulator_inputs, y_model)
print("Fitted emulator kernel:")
print(emulator.kernel_)


In [ ]:
theta_reference = theta_model.mean(axis=0)
emulator_prediction = np.array([
    eta_predict(x, theta_reference, emulator)[0] for x in x_obs
])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_obs, y_obs, "o", ms=4, label="Observations")
ax.plot(x_obs, emulator_prediction, label="Emulator at reference theta")
ax.set(xlabel="Application domain", ylabel="Response", title="Pre-calibration emulator")
ax.legend()
plt.show()


## Run integrated-discrepancy calibration

`run_delta_sampler` estimates a spatially varying correction for each calibration parameter. The returned `delta_chain` has shape `(iterations, parameters, observations)`, while `ell_delta` and `var_delta` contain the final discrepancy GP hyperparameters.

In [ ]:
N_MCMC = 300
BURN_IN = 100

delta_chain, ell_delta, var_delta = run_delta_sampler(
    x_obs=x_obs,
    y_obs=y_obs,
    theta=theta_reference,
    gp_eta=emulator,
    dtheta=theta_model.shape[1],
    n_mcmc=N_MCMC,
    burn_in=BURN_IN,
    mh_scale_delta=0.15,
    allow_singular_covariance=True,
    random_seed=7,
)

posterior_delta = delta_chain[BURN_IN:].mean(axis=0)
calibrated_prediction = np.array([
    eta_predict(x, theta_reference + posterior_delta[:, i], emulator)[0]
    for i, x in enumerate(x_obs)
])

print("delta_chain shape:", delta_chain.shape)
print("Posterior mean parameter corrections:", posterior_delta.mean(axis=1))
print("Final discrepancy length scales:", ell_delta)
print("Final discrepancy variances:", var_delta)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for parameter_index, correction in enumerate(posterior_delta, start=1):
    axes[0].plot(x_obs, correction, label=fr"$\delta_{{\theta_{parameter_index}}}(x)$")
axes[0].axhline(0.0, color="black", lw=0.8)
axes[0].set(xlabel="Application domain", ylabel="Correction", title="Posterior mean discrepancy corrections")
axes[0].legend()

axes[1].plot(x_obs, y_obs, "o", ms=4, label="Observations")
axes[1].plot(x_obs, emulator_prediction, label="Reference emulator")
axes[1].plot(x_obs, calibrated_prediction, label="Calibrated emulator")
axes[1].set(xlabel="Application domain", ylabel="Response", title="Calibration result")
axes[1].legend()
plt.tight_layout()
plt.show()


## Next steps

For an analysis rather than a demonstration, increase `N_MCMC`, inspect trace diagnostics, compare multiple random seeds, and use the validation helpers exported by the package. The existing `config.json` workflow remains available through the `integrated-delta-calibration` command.